## Gemma-2-2bのロード

In [3]:
import torch
from transformer_lens import HookedTransformer

# 1. 高速化設定
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 2. モデルをロード（ここを実行しないと model という名前が生まれません）
model = HookedTransformer.from_pretrained("google/gemma-2-2b", device=device)

/Users/kao/miniforge3/envs/sae2025/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 44.33it/s]


Loaded pretrained model google/gemma-2-2b into HookedTransformer


## Gemma-2-2bのプロンプト

In [9]:
# プロンプト入力
input_text = "I am a steak!"

# AIに続きを書かせる
output = model.generate(input_text, max_new_tokens=50, temperature=0.7)

print("\n--- Gemmaの回答 ---")
print(output)

100%|███████████████████████████████████████████████████████████████████████████████████| 50/50 [00:02<00:00, 18.39it/s]


--- Gemmaの回答 ---
I am a steak! With so many people lately, I feel like I’m at steak heaven. I have had some pretty amazing steaks this past week. I am a little biased though. I think because my husband is my steak master and he was born and raised in


## SAE Lensのロード（ブロックごとに変更）

In [10]:
import os
import torch
from sae_lens import SAE

# 1. トークナイザーの警告を消す（おまじない）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. デバイス設定
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 3. レンズ情報の指定
release = "gemma-scope-2b-pt-res"
sae_id = "layer_20/width_16k/average_l0_71"

print(f"🔬 {sae_id} をロード中...")

# idから場所を勝手に特定する

try:
    layer_num = sae_id.split("/")[0].split("_")[1] # "layer_20" -> "20"
    layer_name = f"blocks.{layer_num}.hook_resid_post"
    print(f"📍 自動特定: これは第 {layer_num} 層用のレンズですね。")
    print(f"   接続先: {layer_name}")
except:
    print("⚠️ 層番号の自動特定に失敗しました。手動設定が必要です。")
    layer_name = "blocks.20.hook_resid_post" # フォールバック

# ----------------------------

# 4. 【重要】3点セットを明示的に取得する新しい書き方
# from_pretrained ではなく、 _with_cfg_and_sparsity を使います
sae, cfg_dict, sparsity = SAE.from_pretrained_with_cfg_and_sparsity(
    release=release,
    sae_id=sae_id,
    device=device
)

print("\n✅ 準備完了！")
print(f"・SAE本体: ロードOK")
print(f"・設定データ: {cfg_dict['d_sae']} 個の特徴量を持っています")
print(f"・疎性データ: L0 = {sparsity}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


🔬 layer_20/width_16k/average_l0_71 をロード中...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: eae8c4dc-0d47-432a-b570-c37837edd39d)')' thrown while requesting HEAD https://huggingface.co/google/gemma-scope-2b-pt-res/resolve/main/layer_20/width_16k/average_l0_71/params.npz
Retrying in 1s [Retry 1/5].



✅ 準備完了！
・SAE本体: ロードOK
・設定データ: 16384 個の特徴量を持っています
・疎性データ: L0 = None


## SAE Lensのロードと結果（ブロックごとに変更）

In [24]:
import torch
import os
from sae_lens import SAE
from transformer_lens import HookedTransformer

# ==========================================
# 1. 解析設定 (Configuration)
# ==========================================
# 解析対象のSAE ID
# ※ここを変更するだけで、自動的に対象の層（Layer）が切り替わります
SAE_RELEASE = "gemma-scope-2b-pt-res"
SAE_ID = "layer_20/width_16k/average_l0_71"

# 解析したいテキスト
INPUT_TEXT = "私はステーキである。"

# ==========================================
# 2. 環境セットアップ (Setup)
# ==========================================
# トークナイザーの並列化によるデッドロック防止（Mac/Linux向け安全策）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# デバイスの自動判定 (Mac MPS / CPU)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🚀 使用デバイス: {device}")

# ==========================================
# 3. 接続ポイントの自動特定 (Auto-Resolution)
# ==========================================
# SAE ID文字列から「層番号」を抽出し、モデル内の正しいフック名を生成します
try:
    # "layer_20/..." -> "20" を抽出
    layer_num = SAE_ID.split("/")[0].split("_")[1]
    HOOK_POINT = f"blocks.{layer_num}.hook_resid_post"
    print(f"📍 解析対象: 第 {layer_num} 層 (Hook Point: {HOOK_POINT})")
except Exception as e:
    print(f"⚠️ 層番号の自動特定に失敗しました。デフォルト値を使用します: {e}")
    HOOK_POINT = "blocks.20.hook_resid_post"

# ==========================================
# 4. モデルとSAEのロード (Loading)
# ==========================================
print(f"🔄 SAE ({SAE_ID}) をロード中...")
sae, cfg_dict, sparsity = SAE.from_pretrained_with_cfg_and_sparsity(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device
)

# Gemmaモデルのロード（既にメモリにある場合はスキップして高速化）
if 'model' not in locals():
    print("🧠 Gemmaモデルをロード中...")
    model = HookedTransformer.from_pretrained("google/gemma-2-2b", device=device)

# ==========================================
# 5. 特徴量解析の実行 (Execution)
# ==========================================
# (1) テキストをモデルに入力し、指定層の生のアクティベーション（脳波）を取得
_, cache = model.run_with_cache(INPUT_TEXT, prepend_bos=True)
original_act = cache[HOOK_POINT]

# (2) SAEを通して、アクティベーションを「意味のある特徴量」に分解
feature_acts = sae.encode(original_act)

#実際に反応した（0より大きい）特徴量の総数を数える
active_count = (feature_acts[0, -1, :] > 0).sum().item()
print(f"📊 全体のアクティブな特徴量数: {active_count} 個 / 16384 個中")
print(f"   (残りの {16384 - active_count} 個は完全に沈黙しています)")

# (3) トップｋ個を表示してみる (ｋ…上から強い順)
top_k = 70
top_values, top_indices = torch.topk(feature_acts[0, -1, :], k=top_k)

    
# ==========================================
# 6. 結果の表示 (Output)
# ==========================================
print(f"\n📖 入力テキスト: 「{INPUT_TEXT}」")
print(f"\n🔬 観察結果 (トップ {top_k}):")
print("-" * 60)

for i in range(top_k):
    feature_id = top_indices[i].item()
    strength = top_values[i].item()
    
    # 強度が弱すぎる（ゴミ）場合はカッコ書きにする演出
    status = "🔥" if strength > 5.0 else "  "
    
    # Neuronpedia用のリンク生成（ID内のスラッシュをハイフンに置換）
    width = "16k" # 今回は16kを使っているため固定
    
    neuronpedia_id = f"{layer_num}-gemmascope-res-{width}"
    url = f"https://www.neuronpedia.org/gemma-2-2b/{neuronpedia_id}/{feature_id}"
    
    print(f"{status} Rank {i+1:<2} | ID: {feature_id:<6} | 強度: {strength:.2f}")
    print(f"   🔗解説リンク {url}")
    print("-" * 60)

    # AIに続きを書かせる
output = model.generate(input_text, max_new_tokens=100, temperature=0.7)

print("\n--- Gemmaの回答 ---")
print(output)

🚀 使用デバイス: mps
📍 解析対象: 第 20 層 (Hook Point: blocks.20.hook_resid_post)
🔄 SAE (layer_20/width_16k/average_l0_71) をロード中...
📊 全体のアクティブな特徴量数: 61 個 / 16384 個中
   (残りの 16323 個は完全に沈黙しています)

📖 入力テキスト: 「私はステーキである。」

🔬 観察結果 (トップ 70):
------------------------------------------------------------
🔥 Rank 1  | ID: 4945   | 強度: 81.88
   🔗解説リンク https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/4945
------------------------------------------------------------
🔥 Rank 2  | ID: 7719   | 強度: 69.43
   🔗解説リンク https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/7719
------------------------------------------------------------
🔥 Rank 3  | ID: 9768   | 強度: 36.33
   🔗解説リンク https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/9768
------------------------------------------------------------
🔥 Rank 4  | ID: 10788  | 強度: 31.32
   🔗解説リンク https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/10788
------------------------------------------------------------
🔥 Rank 5  | ID: 6631   | 強度:

100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [00:04<00:00, 22.21it/s]


--- Gemmaの回答 ---
I am a steak!
I am a steak,
And I’m a big steak
And I’m a big fat steak!

Do you remember the above song from the old Disney film The Jungle Book? It’s one of my favorite songs and I’ve always loved it. One of my brothers was a huge fan of the movie and would keep me up at night singing the song. I always loved the song and I was so excited to see that Disney released a new version of it. The
